# 2D板の弾性波伝播: ケース比較動画

GitHubのリポジトリをcloneし、裏でシミュレーションを実行して、複数ケースを縦に並べた比較動画(mp4)を表示します。
本体は Python 3.14 が必要なため、Colab標準のPythonではなく `uv` が用意する3.14の仮想環境で実行します。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mr-Kondo/Elastic_Wave_Propagate_on_the_2D_PLATE/blob/main/notebooks/colab_compare.ipynb)

In [ ]:
#@title 設定
PRESET = "boundaries"  #@param ["boundaries", "poisson", "integrators"]
STEPS = 560  #@param {type:"integer"}
REF = "main"  #@param {type:"string"}  # ブランチまたはタグ(例: v0.4.0)

In [ ]:
#@title 1. リポジトリ取得と環境構築(初回は数分)
import subprocess

REPO = "https://github.com/Mr-Kondo/Elastic_Wave_Propagate_on_the_2D_PLATE.git"
subprocess.run(["rm", "-rf", "/content/ewp"], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO, "/content/ewp"], check=True)
subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--locked", "--no-dev"], cwd="/content/ewp", check=True)

In [ ]:
#@title 2. 裏で実行(ログを表示)
import subprocess, time

if "job" in globals() and job.poll() is None:
    raise RuntimeError("前回のジョブがまだ実行中です。終わるまで待ってから再実行してください")
cmd = ["uv", "run", "--locked", "--no-dev", "python", "run_compare.py",
       "--preset", PRESET, "--steps", str(STEPS), "--out", "feedback/compare"]
with open("/content/ewp/compare.log", "w") as log:
    job = subprocess.Popen(cmd, cwd="/content/ewp", stdout=log, stderr=subprocess.STDOUT)
shown = 0
try:
    while True:
        done = job.poll() is not None
        lines = open("/content/ewp/compare.log").read().splitlines()
        print("\n".join(lines[shown:]), end="\n" if lines[shown:] else "")
        shown = len(lines)
        if done:
            break
        time.sleep(2)
finally:
    if job.poll() is None:  # セルを中断したら子プロセスも止める
        job.terminate()
        job.wait()
assert job.returncode == 0, f"failed with exit code {job.returncode}"
out = lines[-1].removeprefix("wrote ")  # 今回の出力ファイル

In [ ]:
#@title 3. 比較動画を表示
from pathlib import Path
from IPython.display import Image, Video, display

out_path = Path("/content/ewp") / out
display(Video(out_path, embed=True, width=640) if out_path.suffix == ".mp4" else Image(filename=str(out_path)))